# CARA v6.3 — Concept–Token Attention Region Attack

Independent research implementation, designed like **MPCAttack**: normal Python classes and explicit method calls. No `exec`, source-code strings, code hashes, notebook imports, or hidden code loading.

**Run in order:** Config → Package → Data → Preprocessing → Feature Extractors → Loss Function → Attack → Evaluation → Visualization → Pilot.

**Research provenance:** FINER object prompts and helper definitions; SAGA attention aggregation; Grad-ECLIP dense encoder; MPC's InfoNCE direction and momentum sign updates. All CARA-specific modifications are labeled. Original paper functions appear as regular Python code below.

**Source repositories:** [MPCAttack](https://github.com/LiYuanBoJNU/MPCAttack), [FINER](https://github.com/ExplainableML/finer), [SAGA](https://github.com/jaehyun-kwak/SAGA). Paper-derived and CARA-original functions are separated by cell titles.


## Config

In [ ]:
from pathlib import Path

# Execution
INSTALL_PACKAGES = True
RUN_ATTACK = True
RUN_EVALUATION = True
RUN_VISUALIZATION = True
NUM_SAMPLES = 5
START_INDEX = 0
SEED = 2026

# Dataset: attach https://www.kaggle.com/datasets/zintom/blackbox-adversarial-attack-dataset
DATA_ROOT = Path("/kaggle/input/blackbox-adversarial-attack-dataset")
SOURCE_DIR = None                       # Auto-detect a directory named bigscale
TARGET_DIR = None                       # Auto-detect a directory named target_images
PAIR_MANIFEST = None                    # Optional CSV with columns source,target
OUTPUT_DIR = Path("/kaggle/working/cara_v6_3")

# Hypothesis 2
K = 5
STEPS = 300
STAGE_STEPS = 30
RANDOM_START = 150
XAI_METHOD = "grad_eclip"               # grad_eclip | qwen_attention
EPSILON = 16.0                          # In [0, 255] pixel coordinates
ALPHA = 1.0
MOMENTUM = 0.9
TEMPERATURE = 0.2
OMEGA = 2.0
STRICT_CROP_MASK = True                 # CARA change, not MPC's default
SAVE_EVERY = 30
ATTENTION_LAYER = 29
NEGATIVE_RETRIES = 10
CAPTION_ATTEMPTS = 40

# Models
IMAGE_SIZE = 224
CLIP_G14_ID = "laion/CLIP-ViT-g-14-laion2B-s12B-b42K"
CLIP_B16_ID = "openai/clip-vit-base-patch16"
CLIP_B32_ID = "openai/clip-vit-base-patch32"
QWEN_ID = "Qwen/Qwen3-VL-8B-Instruct"
EVALUATION_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
CLIP_DEVICES = {"G14": "cuda:1", "B16": "cuda:0", "B32": "cuda:0"}
ATTACK_DEVICE = "cuda:0"
QWEN_MAX_MEMORY_GIB = {0: 12, 1: 12}  # Adjust for your Kaggle GPUs
IMAGE_WEIGHT = 0.6
TEXT_WEIGHT = 0.4
CAPTION_TEMPERATURE = 0.8
CAPTION_TOP_P = 0.9

# Evaluation is a caption-based proxy until human labels are completed.
EVALUATION_PROMPT = "Describe this image in one concise sentence, no longer than 20 words."


## Package

In [ ]:
# Do not reinstall torch and torchvision: Kaggle supplies CUDA-compatible builds.
if INSTALL_PACKAGES:
    import subprocess
    import sys

    packages = [
        "transformers==4.57.1",
        "accelerate==1.11.0",
        "qwen-vl-utils==0.0.14",
        "ftfy==6.3.1",
        "regex",
        "pandas",
        "matplotlib",
        "scikit-learn",
        "loguru",
        "git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
    print("Dependencies installed. Restart kernel if core packages were previously imported.")


In [ ]:
import gc
import json
import math
import random
import re
from collections.abc import Sequence
from contextlib import redirect_stdout
from io import StringIO

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image, ImageOps
from torch import Tensor
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def write_json(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def image_to_tensor(image):
    data = np.asarray(image.convert("RGB"), dtype=np.uint8).copy()
    return torch.from_numpy(data).permute(2, 0, 1).unsqueeze(0).float()


def tensor_to_image(tensor):
    tensor = tensor.detach().cpu().squeeze(0).clamp(0, 255).round().byte()
    return Image.fromarray(tensor.permute(1, 2, 0).numpy())


def l2_normalize(x):
    return F.normalize(x.float(), dim=-1)


set_seed(SEED)
print("CUDA GPUs:", torch.cuda.device_count())


## Data

In [ ]:
class ImagePairDataset:
    def __init__(self, root, source_dir=None, target_dir=None, manifest=None):
        self.root = Path(root)
        self.source_dir = source_dir
        self.target_dir = target_dir
        self.manifest = manifest
        self.transform = transforms.Compose([
            transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(IMAGE_SIZE),
        ])

    def find_directory(self, name, override):
        if override is not None:
            directory = Path(override)
            if not directory.is_dir():
                raise FileNotFoundError(directory)
            return directory

        matches = [p for p in self.root.rglob(name) if p.is_dir()]
        if self.root.name == name:
            matches.insert(0, self.root)
        if len(matches) != 1:
            raise ValueError(f"Expected one {name} folder; found: {matches}")
        return matches[0]

    def get_pairs(self, start=0, count=5):
        if self.manifest is not None:
            table = pd.read_csv(self.manifest)
            if not {"source", "target"}.issubset(table.columns):
                raise ValueError("PAIR_MANIFEST needs source,target columns")
            base = Path(self.manifest).parent
            pairs = []
            for row in table.itertuples(index=False):
                source = Path(row.source)
                target = Path(row.target)
                pairs.append((source if source.is_absolute() else base / source,
                              target if target.is_absolute() else base / target))
        else:
            source_set = ImageFolder(self.find_directory("bigscale", self.source_dir))
            target_set = ImageFolder(self.find_directory("target_images", self.target_dir))
            if len(source_set) != len(target_set):
                raise ValueError("Source and target counts differ; use PAIR_MANIFEST")
            pairs = list(zip([row[0] for row in source_set.samples],
                             [row[0] for row in target_set.samples]))

        selected = pairs[start:start + count]
        if self.manifest is None:
            matches = sum(
                Path(source).stem.split("_")[0] == Path(target).stem.split("_")[0]
                for source, target in selected
            )
            if matches != len(selected):
                print("WARNING: File IDs may not match. Inspect the pair preview or supply PAIR_MANIFEST.")
        if len(selected) != count:
            raise ValueError(f"Requested {count} pairs, got {len(selected)}")
        for source, target in selected:
            if not Path(source).is_file() or not Path(target).is_file():
                raise FileNotFoundError(f"Missing source/target: {source}, {target}")
            if Path(source).resolve() == Path(target).resolve():
                raise ValueError("Source and target cannot be the same file")
        return selected

    def load_image(self, path):
        with Image.open(path) as handle:
            image = ImageOps.exif_transpose(handle).convert("RGB")
        return self.transform(image)


DATASET = ImagePairDataset(DATA_ROOT, SOURCE_DIR, TARGET_DIR, PAIR_MANIFEST)


## Preprocessing

### FINER — Original object and negative-object prompts

In [ ]:
SYSTEM_MSG_OBJECTS = (
    "You are an information extraction assistant.\n"
    "From the caption, select up to FIVE main objects. A main object must have at least one descriptive attribute in the caption "
    "(e.g., color, size, material, possession/with-phrase, appositive, relative clause, or an explicit number).\n"
    "Rules:\n"
    "• Output only object names that appear in the caption and are part of the described scene. Never invent or infer.\n"
    "• Prefer plain object names in the output (omit adjectives), but KEEP explicit numbers/quantifiers if the caption states them "
    "(e.g., 'two cats', 'five chickens', 'a pair of skis').\n"
    "• If multiple mentions share the same name and no explicit number is given, output the plural form (e.g., 'dogs'). \n"
    "• List 1–5 main objects; if only one is present, output just that one.\n"
    "Do not add quantifiers like 'some' unless present in the caption.\n"
    "Format:\n"
    "Return EXACTLY one line:\n"
    "PHRASE=<comma-separated list with 'and' before the last item, e.g., 'a dog, a cat and two birds'>\n"
    "No trailing period. No extra text."
)

FEW_SHOT_OBJECTS = [
    (
        "A bustling street scene with three blue buses at the curb, two red cars in traffic, "
        "a tall glass office building behind them, and a bronze statue with a raised arm in the plaza.",
        "PHRASE=three buses, two cars, an office building and a statue"
    ),
    (
        "On a wooden table, a cast-iron skillet sits beside a blue ceramic vase of sunflowers, "
        "with red apples and green pears piled high in a stainless steel bowl.",
        "PHRASE=a skillet, a vase, apples, pears and a bowl"
    ),
]

SYSTEM_MSG_NEGOBJ = (
    "You are a negative object creator.\n"
    "You will receive a caption, an object list as PHRASE=..., and REPLACE_INDEX=k (1-based).\n"
    "Replace EXACTLY the k-th object in PHRASE with a distinctly different NEGATIVE object.\n"
    "\n"
    "Keep ALL other objects unchanged and preserve their order and punctuation. Keep the same quantifier/determiner "
    "for the replaced slot (e.g., 'two cats' -> 'two bicycles').\n"
    "\n"
    "Constraints for the NEGATIVE object:\n"
    "• It must be distinctly different from the replaced object (not a synonym; not just singular/plural).\n"
    "• It must NOT be a synonym or near-equivalent of ANY object that appears in the caption.\n"
    "• It must NOT appear anywhere in the caption (as a whole word, singular or plural).\n"
    "• Do not modify any other items; do not reorder items; do not add or remove items.\n"
    "\n"
    "Edge cases (must follow):\n"
    "• If REPLACE_INDEX is greater than the number of objects in PHRASE, replace the LAST object.\n"
    "• If REPLACE_INDEX is less than 1, replace the FIRST object.\n"
    "\n"
    "Self-check (must hold):\n"
    "• Same number of items as input; exactly one item (the k-th per the rule above) differs.\n"
    "\n"
    "Output EXACTLY one line:\n"
    "PHRASE=<the new list, same format>\n"
    "No extra text. No quotes. No trailing period."
)

# Few-shots cover different list lengths and positions; each uses REPLACE_INDEX to avoid a last-item bias.
FEW_SHOT_NEGOBJ = [
    (
        # Replace FIRST of three
        "Caption:\nA small yellow school bus is parked next to two red cars in front of a brick office building.\n"
        "PHRASE=a bus, two cars and an office building\n"
        "REPLACE_INDEX=1",
        "PHRASE=a boat, two cars and an office building"
    ),
    (
        "Caption:\nTwo hikers stand on a wooden bridge with their dog beside them over a calm river.\n"
        "PHRASE=two hikers, a dog, a bridge and a river\n"
        "REPLACE_INDEX=2",
        "PHRASE=two hikers, a bicycle, a bridge and a river"
    ),
    (
        "Caption:\nOn a side table there is a bowl of fruit, a ceramic vase, and a brass lamp.\n"
        "PHRASE=a bowl of fruit, a vase and a lamp\n"
        "REPLACE_INDEX=3",
        "PHRASE=a bowl of fruit, a vase and a clock"
    ),
    (
        "Caption:\nA dashboard card labeled 'Latency' shows a graph with a menu icon and a legend for several lines.\n"
        "PHRASE=a card, a graph, a menu icon, a legend and lines\n"
        "REPLACE_INDEX=3",
        "PHRASE=a card, a graph, a slider, a legend and lines"
    ),
    (
        "Caption:\nA booking screen shows a highlighted 12:05–12:30 time slot, a search box, an entry for Animal Kingdom Villas, a restaurant icon, and a market link.\n"
        "PHRASE=a time slot, a box, an entry, a restaurant and a market\n"
        "REPLACE_INDEX=4",
        "PHRASE=a time slot, a box, an entry, a ticket counter and a market"
    ),
]

CATEGORIES = ["natural_image", "chart_graph", "screenshot_ui", "document_text"]

SYSTEM_MSG_CATEGORY = (
    "You are a strict classifier. Based ONLY on the caption text, assign EXACTLY ONE label.\n"
    "Allowed labels:\n"
    "  - natural_image\n"
    "  - chart_graph   (includes diagrams, flowcharts, schematics, illustrations)\n"
    "  - screenshot_ui (UI/app/website/IDE/terminal screenshots)\n"
    "  - document_text (scans, printed pages, receipts, invoices, whiteboards, handwriting)\n"
    "\n"
    "Rules:\n"
    "• Return ONLY the label string, nothing else.\n"
    "• If multiple seem plausible, pick the single best one.\n"
    "• If unsure, choose natural_image.\n"
)

FEW_SHOT_CATEGORY = [
    ("A line chart comparing quarterly revenue for three products with a legend and axes labels.",
     "chart_graph"),
    ("A screenshot of a mobile banking app showing the account balance and a transfer button.",
     "screenshot_ui"),
    ("A scanned receipt from a grocery store listing items and totals.",
     "document_text"),
    ("A photo of a dog running on a beach at sunset.",
     "natural_image"),
    ("A flowchart illustrating an authentication pipeline with decision nodes and arrows.",
     "chart_graph"),
]


### FINER — Original prompt and parsing helpers

The functions below reproduce the original FINER helpers as ordinary readable Python definitions. Only aliases required by its original package-relative imports are provided.

In [ ]:
import re
from typing import List

def truncate_first_tokens(tokenizer, text: str, max_tokens: int = 512) -> str:
    """
    Keep only the first `max_tokens` tokenizer tokens.
    """
    ids = tokenizer(text or "", add_special_tokens=False).input_ids
    ids = ids[:max_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def parse_tagged_output(
    text: str,
    *,
    allowed_tags: Sequence[str],
    max_chars: int = 512,
    split_on_semicolon: bool = False,
    strip_trailing_pattern: str = r"[.;:\s]+$",
) -> str:
    """
    Parse a generated single-line output like:
      PHRASE=...
      SENTENCE=...
      ACC_PHRASE=...
    by scanning for the first matching tagged line.

    Fallback:
      - if no tagged line is found, use the first non-empty line.

    Normalization:
      - strip outer quotes
      - optional split on first ';'
      - strip trailing punctuation via regex
      - collapse internal whitespace
      - clip to max_chars
    """
    raw = (text or "").strip()
    lines = [
        line.strip()
        for line in raw.splitlines()
        if line.strip() and line.strip() not in {"```", "~~~"}
    ]

    if not lines:
        return ""

    tag_pattern = "|".join(re.escape(tag) for tag in allowed_tags)
    pattern = re.compile(
        rf"^\s*(?:{tag_pattern})\s*[:=]\s*(.+)$",
        flags=re.IGNORECASE,
    )

    value = ""
    for line in lines:
        match = pattern.match(line)
        if match:
            value = match.group(1).strip()
            break

    if not value:
        value = lines[0].strip()

    value = (
        value.strip()
        .strip('"')
        .strip("'")
        .strip("“”‘’")
        .strip()
    )

    if split_on_semicolon:
        value = value.split(";", 1)[0].strip()

    if strip_trailing_pattern:
        value = re.sub(strip_trailing_pattern, "", value)

    value = re.sub(r"\s+", " ", value).strip()

    if max_chars is not None:
        value = value[:max_chars]

    return value


_truncate_caption_first_tokens = truncate_first_tokens
def _split_objects_phrase(phrase: str) -> List[str]:
    """
    Split 'a dog, a cat and two birds' -> ['a dog','a cat','two birds'].
    Heuristic: split on the last ' and ' and commas on the left.
    Used only for lightweight validation; storage remains a single string.
    """
    s = (phrase or "").strip()
    if not s:
        return []
    if " and " in s:
        left, last = s.rsplit(" and ", 1)
        left_parts = [p.strip() for p in left.split(",") if p.strip()]
        items = left_parts + [last.strip()]
    else:
        items = [p.strip() for p in s.split(",") if p.strip()]
    return items

def _messages_objects(caption_text: str) -> List[dict]:
    """Chat messages for the 'objects' mode with few-shot."""
    msgs = [{"role": "system", "content": SYSTEM_MSG_OBJECTS}]

    for cap, ans in FEW_SHOT_OBJECTS:
        demo_user = (
            "Caption:\n" + cap + "\n\n"
            "Task: From this caption, select up to five main objects that have at least one descriptive attribute in the caption. "
            "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "
            "Return EXACTLY one line:\nPHRASE=<...>"
        )
        msgs.append({"role": "user", "content": demo_user})
        msgs.append({"role": "assistant", "content": ans})

    task_user = (
        "Caption:\n" + caption_text + "\n\n"
        "Task: Select up to five main objects that have at least one descriptive attribute in the caption. "
        "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "
        "Return EXACTLY one line:\nPHRASE=<...>"
    )
    msgs.append({"role": "user", "content": task_user})
    return msgs

def build_objects_prompt(tokenizer, caption: str) -> str:
    """Final chat prompt using the FIRST 512 caption tokens."""
    cap_trunc = _truncate_caption_first_tokens(tokenizer, caption, 512)
    return tokenizer.apply_chat_template(
        _messages_objects(cap_trunc),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

def parse_objects_output(text: str) -> str:
    return parse_tagged_output(
        text,
        allowed_tags=("PHRASE",),
        max_chars=512,
    )


def _validate_single_replacement(original_phrase: str, new_phrase: str) -> bool:
    """
    Light sanity: same number of items and exactly one item differs (string-compare after strip).
    """
    old = _split_objects_phrase(original_phrase)
    new = _split_objects_phrase(new_phrase)
    if not old or not new or len(old) != len(new):
        return False
    diffs = sum(1 for a, b in zip(old, new) if a.strip() != b.strip())
    return diffs == 1


def _messages_negobj(caption_text: str, obj_phrase: str, replace_index_1based: int) -> List[dict]:
    """
    Chat messages for negative replacement with an explicit slot to replace.
    The model must replace exactly REPLACE_INDEX (1-based).
    """
    msgs = [{"role": "system", "content": SYSTEM_MSG_NEGOBJ}]
    for demo_user, demo_assistant in FEW_SHOT_NEGOBJ:
        msgs.append({"role": "user", "content": demo_user})
        msgs.append({"role": "assistant", "content": demo_assistant})
    task_user = f"Caption:\n{caption_text}\nPHRASE={obj_phrase}\nREPLACE_INDEX={replace_index_1based}"
    msgs.append({"role": "user", "content": task_user})
    return msgs


def build_negobj_prompt(tokenizer, caption: str, obj_phrase: str, replace_index_1based: int) -> str:
    """Final chat prompt for negative replacement, with an explicit index (1-based) to replace."""
    cap_trunc = _truncate_caption_first_tokens(tokenizer, caption, 512)
    return tokenizer.apply_chat_template(
        _messages_negobj(cap_trunc, obj_phrase, replace_index_1based),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


### CARA — Caption bank and object replacement

**CARA extension:** FINER generates a negative `PHRASE=...`, not a complete negative caption. The caption reconstruction and hard-negative selection below are new CARA integration code; they are not attributed to FINER.

In [ ]:
class CaptionBank:
    def __init__(self, captioner, k=5):
        self.captioner = captioner
        self.k = k

    def generate_captions(self, image):
        captions = []
        for _ in range(CAPTION_ATTEMPTS):
            caption = self.captioner.caption(image, sample=True).strip()
            if caption.casefold() not in {x.casefold() for x in captions}:
                captions.append(caption)
            if len(captions) == self.k:
                break
        if len(captions) != self.k:
            raise RuntimeError(f"Only {len(captions)} distinct captions; need {self.k}")
        return captions

    def extract_objects(self, caption):
        prompt = build_objects_prompt(self.captioner.processor.tokenizer, caption)
        response = self.captioner.generate(prompt, max_new_tokens=80)
        phrase = parse_objects_output(response)
        objects = _split_objects_phrase(phrase)
        if not (1 <= len(objects) <= 5):
            raise ValueError(f"Invalid FINER object list: {response}")
        return {"caption": caption, "phrase": phrase, "objects": objects}

    @staticmethod
    def reconstruct_caption(caption, old_item, new_item):
        old_words = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", old_item)
        new_words = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", new_item)
        common = 0
        while common < min(len(old_words), len(new_words)):
            if old_words[common].lower() != new_words[common].lower():
                break
            common += 1
        old_core = old_words[common:]
        new_core = new_words[common:]
        if not old_core or not new_core:
            return None

        for index in range(len(old_core)):
            old = " ".join(old_core[index:])
            new = " ".join(new_core[index:]) if len(new_core) > index else " ".join(new_core)
            pattern = re.compile(r"\b" + re.escape(old).replace(r"\ ", r"\s+") + r"\b", re.I)
            hits = list(pattern.finditer(caption))
            if len(hits) == 1 and new:
                hit = hits[0]
                changed = caption[:hit.start()] + new + caption[hit.end():]
                if changed.casefold() != caption.casefold():
                    return changed
        return None

    def generate_negative(self, object_record):
        caption = object_record["caption"]
        phrase = object_record["phrase"]
        objects = object_record["objects"]
        for _ in range(NEGATIVE_RETRIES):
            index = random.randint(1, len(objects))  # Uniform object slot, as in FINER
            prompt = build_negobj_prompt(self.captioner.processor.tokenizer, caption, phrase, index)
            raw = self.captioner.generate(prompt, max_new_tokens=80)
            new_phrase = parse_objects_output(raw)
            new_objects = _split_objects_phrase(new_phrase)
            valid = (
                _validate_single_replacement(phrase, new_phrase)
                and len(new_objects) == len(objects)
                and all(objects[j] == new_objects[j] for j in range(len(objects)) if j != index - 1)
            )
            if not valid:
                continue
            new_caption = self.reconstruct_caption(caption, objects[index - 1], new_objects[index - 1])
            if new_caption and new_caption.casefold() != caption.casefold():
                return {"caption": new_caption, "phrase": new_phrase, "replaced_position": index}
        raise RuntimeError(f"No validated FINER negative for: {caption}")

    def create(self, source_image, target_image):
        source = self.generate_captions(source_image)
        target = self.generate_captions(target_image)
        object_records = [self.extract_objects(caption) for caption in source]
        target_records = [self.extract_objects(caption) for caption in target]
        negatives = [self.generate_negative(record) for record in object_records]
        negative_captions = [record["caption"] for record in negatives]
        if len(set(x.casefold() for x in negative_captions)) != self.k:
            raise ValueError("Negative captions contain duplicates")

        source_concepts = list(dict.fromkeys(obj for record in object_records for obj in record["objects"]))
        target_concepts = list(dict.fromkeys(obj for record in target_records for obj in record["objects"]))
        return {
            "source_captions": source,
            "target_captions": target,
            "negative_captions": negative_captions,
            "source_concepts": source_concepts,
            "target_concepts": target_concepts,
            "source_objects": object_records,
            "target_objects": target_records,
            "negative_details": negatives,
        }


### XAI — Spatial crop and dispersion (CARA)

In [ ]:
class RegionSelector:
    def __init__(self, height=224, width=224):
        self.height = height
        self.width = width

    @staticmethod
    def stage_area(step):
        stage = (step - 1) // STAGE_STEPS
        return (9 + stage) / 18

    def select(self, heatmap, step):
        area = self.stage_area(step)
        height = max(1, min(self.height, round(self.height * math.sqrt(area))))
        width = max(1, min(self.width, round(self.width * math.sqrt(area))))
        if step >= RANDOM_START:
            top = random.randint(0, self.height - height)
            left = random.randint(0, self.width - width)
            return top, left, height, width, "random"
        if height == self.height and width == self.width:
            return 0, 0, height, width, "xai"

        heat = F.interpolate(heatmap[None, None].float(), size=(self.height, self.width),
                             mode="bilinear", align_corners=False)[0, 0]
        integral = F.pad(heat.double(), (1, 0, 1, 0)).cumsum(0).cumsum(1)
        scores = (integral[height:, width:] - integral[:-height, width:]
                  - integral[height:, :-width] + integral[:-height, :-width])
        index = int(scores.flatten().argmax())
        top, left = divmod(index, self.width - width + 1)
        return top, left, height, width, "xai"


class XAIMap:
    @staticmethod
    def normalize(maps):
        valid = {}
        for name, value in maps.items():
            value = torch.as_tensor(value).float().cpu()
            if value.ndim != 2 or not torch.isfinite(value).all():
                continue
            if value.min() < 0 or value.max() - value.min() <= 1e-8:
                continue
            small = F.interpolate(value[None, None], (32, 32), mode="bilinear",
                                  align_corners=False)[0, 0]
            if small.max() - small.min() <= 1e-8:
                continue
            valid[name] = small / small.sum()
        if not valid:
            raise ValueError("No informative XAI heatmap")
        return torch.stack(list(valid.values())).mean(0), valid

    @staticmethod
    def dispersion(heatmap):
        probability = heatmap / heatmap.sum()
        positive = probability[probability > 0]
        entropy = -(positive * positive.log()).sum()
        yy, xx = torch.meshgrid(torch.linspace(0, 1, probability.shape[0]),
                                torch.linspace(0, 1, probability.shape[1]), indexing="ij")
        cy = (probability * yy).sum()
        cx = (probability * xx).sum()
        variance = (probability * ((yy - cy).square() + (xx - cx).square())).sum()
        return {
            "entropy": float(entropy / math.log(probability.numel())),
            "effective_support": float(entropy.exp() / probability.numel()),
            "variance": float(variance),
        }


### Grad-ECLIP — Original dense vision encoder and map functions

Direct definitions from the referenced Grad-ECLIP implementation. `clipmodel`, `clip_inres`, and `clip_ksize` are assigned by the wrapper class, rather than dynamically injecting code.

In [ ]:
# Set by GradEclipXAI.__init__ before these original functions are invoked.
clipmodel = None
clip_inres = None
clip_ksize = None

def attention_layer(q, k, v, num_heads=1):
    "Compute 'Scaled Dot Product Attention'"
    tgt_len, bsz, embed_dim = q.shape
    head_dim = embed_dim // num_heads
    scaling = float(head_dim) ** -0.5
    q = q * scaling

    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    v = v.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    attn_output_weights = F.softmax(attn_output_weights, dim=-1)
    attn_output_heads = torch.bmm(attn_output_weights, v)
    assert list(attn_output_heads.size()) == [bsz * num_heads, tgt_len, head_dim]
    attn_output = attn_output_heads.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)
    attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, -1)
    attn_output_weights = attn_output_weights.sum(dim=1) / num_heads
    return attn_output, attn_output_weights

def clip_encode_dense(x,n):
    vision_width = clipmodel.visual.transformer.width
    vision_heads = vision_width // 64
    print("[vision_width and vision_heads]:", vision_width, vision_heads)

    # modified from CLIP
    x = x.half()
    x = clipmodel.visual.conv1(x)  
    feah, feaw = x.shape[-2:]

    x = x.reshape(x.shape[0], x.shape[1], -1) 
    x = x.permute(0, 2, 1) 
    class_embedding = clipmodel.visual.class_embedding.to(x.dtype)
    x = torch.cat([class_embedding + torch.zeros(x.shape[0], 1, x.shape[-1]).to(x), x], dim=1)

    ## scale position embedding as the image w-h ratio
    pos_embedding = clipmodel.visual.positional_embedding.to(x.dtype)
    tok_pos, img_pos = pos_embedding[:1, :], pos_embedding[1:, :]
    pos_h = clip_inres // clip_ksize[0]
    pos_w = clip_inres // clip_ksize[1]
    assert img_pos.size(0) == (pos_h * pos_w), f"the size of pos_embedding ({img_pos.size(0)}) does not match resolution shape pos_h ({pos_h}) * pos_w ({pos_w})"
    img_pos = img_pos.reshape(1, pos_h, pos_w, img_pos.shape[1]).permute(0, 3, 1, 2)
    print("[POS shape]:", img_pos.shape, (feah, feaw))
    img_pos = torch.nn.functional.interpolate(img_pos, size=(feah, feaw), mode='bicubic', align_corners=False)
    img_pos = img_pos.reshape(1, img_pos.shape[1], -1).permute(0, 2, 1)
    pos_embedding = torch.cat((tok_pos[None, ...], img_pos), dim=1)
    x = x + pos_embedding
    x = clipmodel.visual.ln_pre(x)

    x = x.permute(1, 0, 2)  # NLD -> LND
    x = torch.nn.Sequential(*clipmodel.visual.transformer.resblocks[:-n])(x)

    attns = []
    atten_outs = []
    vs = []
    qs = []
    ks = []
    for TR in clipmodel.visual.transformer.resblocks[-n:]:
        x_in = x
        x = TR.ln_1(x_in)
        linear = torch._C._nn.linear    
        q, k, v = linear(x, TR.attn.in_proj_weight, TR.attn.in_proj_bias).chunk(3, dim=-1)
        attn_output, attn = attention_layer(q, k, v, 1)  # vision_heads=1
        attns.append(attn)
        atten_outs.append(attn_output)
        vs.append(v)
        qs.append(q)
        ks.append(k)

        x_after_attn = linear(attn_output, TR.attn.out_proj.weight, TR.attn.out_proj.bias)       
        x = x_after_attn + x_in
        x = x + TR.mlp(TR.ln_2(x))

    x = x.permute(1, 0, 2)  # LND -> NLD
    x = clipmodel.visual.ln_post(x)
    x = x @ clipmodel.visual.proj
    return x, x_in, vs, qs, ks, attns, atten_outs, (feah, feaw)

def sim_qk(q, k):
    q_cls = F.normalize(q[:1,0,:], dim=-1) 
    k_patch = F.normalize(k[1:,0,:], dim=-1)

    cosine_qk = (q_cls * k_patch).sum(-1) 
    cosine_qk_max = cosine_qk.max(dim=-1, keepdim=True)[0]
    cosine_qk_min = cosine_qk.min(dim=-1, keepdim=True)[0]
    cosine_qk = (cosine_qk-cosine_qk_min) / (cosine_qk_max-cosine_qk_min)
    return cosine_qk

def grad_eclip(c, qs, ks, vs, attn_outputs, map_size):
    ## gradient on last attention output
    tmp_maps = []
    for q, k, v, attn_output in zip(qs, ks, vs, attn_outputs):
        grad = torch.autograd.grad(
            c,
            attn_output,
            retain_graph=True)[0]

        grad_cls = grad[:1,0,:]
        v_patch = v[1:,0,:]
        cosine_qk = sim_qk(q, k).reshape(-1)
        tmp_maps.append((grad_cls * v_patch * cosine_qk[:,None]).sum(-1)) 

    emap = F.relu_(torch.stack(tmp_maps, dim=0)).sum(0)
    return emap.reshape(*map_size)


### Grad-ECLIP — Class

In [ ]:
class GradEclipXAI:
    def __init__(self, device):
        import clip
        global clipmodel, clip_inres, clip_ksize

        self.device = torch.device(device)
        self.model, _ = clip.load("ViT-B/16", device=self.device, jit=False)
        self.model.eval().requires_grad_(False)
        self.clip = clip

        clipmodel = self.model
        clip_inres = int(self.model.visual.input_resolution)
        clip_ksize = self.model.visual.conv1.kernel_size

    def maps(self, image, concepts):
        if not concepts:
            raise ValueError("No FINER concepts")
        patch_h, patch_w = clip_ksize
        width, height = image.size
        new_width = max(patch_w, round(width / patch_w) * patch_w)
        new_height = max(patch_h, round(height / patch_h) * patch_h)
        preprocess = transforms.Compose([
            transforms.Resize((new_height, new_width)),
            transforms.ToTensor(),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                                 (0.26862954, 0.26130258, 0.27577711)),
        ])
        pixels = preprocess(image).unsqueeze(0).to(self.device).requires_grad_(True)
        tokens = self.clip.tokenize(concepts).to(self.device)
        with torch.no_grad():
            text_features = l2_normalize(self.model.encode_text(tokens))
        with torch.enable_grad(), redirect_stdout(StringIO()):
            output, _, values, queries, keys, _, attention_outputs, grid = clip_encode_dense(pixels, 1)
            scores = l2_normalize(output[:, 0]) @ text_features.T
            maps = {
                name: grad_eclip(scores[0, i], queries, keys, values, attention_outputs, grid)
                for i, name in enumerate(concepts)
            }
        return {key: value.detach().float().cpu() for key, value in maps.items()}, {"model": "Grad-ECLIP"}


### SAGA — Original generation-attention helpers

Direct source functions from the official SAGA implementation. No import from `saga` and no `exec`.

In [ ]:
import os
from loguru import logger

DEBUG = os.environ.get("DEBUG", "0") == "1"

def get_token_indices(decoded_text: str, tokenizer, outputs: dict, inputs: dict, vlm_type="qwen3_vl") -> dict:
    """Extract indices for vision tokens, prompt tokens, and output tokens."""
    if vlm_type == "qwen3_vl":
        marker_start = "<|vision_start|>"
        marker_end = "<|vision_end|>"

    len_prompt_tokens = len(inputs["input_ids"][0])
    len_all_tokens = len(outputs["sequences"][0])

    # +1 to include <|vision_start|> token
    vision_token_start = (
        len(tokenizer(decoded_text.split(marker_start)[0], return_tensors="pt")["input_ids"][0]) + 1
    )
    vision_token_end = len(tokenizer(decoded_text.split(marker_end)[0], return_tensors="pt")["input_ids"][0])

    return {
        "vision_start": vision_token_start,
        "vision_end": vision_token_end,
        "output_start": len_prompt_tokens,
        "output_end": len_all_tokens,
        "output_len": len_all_tokens - len_prompt_tokens,
    }

def aggregate_llm_attention(attn: Tensor, layer_idx: int | None = None) -> Tensor:
    """
    Extract average attention vector from attention outputs.

    Nullifies attention over the first token (<bos>), which is a common practice
    for null attention (https://aclanthology.org/W19-4808.pdf).

    Args:
        attn: Attention tensor from model outputs
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
    """
    if layer_idx is not None and layer_idx != -100:
        layer = attn[layer_idx]
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        vec = torch.concat(
            [
                torch.tensor([0.0]),  # Zero out null attention on <bos>
                attns_per_head[-1][1:].cpu(),
                torch.tensor([0.0]),  # Zero for final generated token
            ]
        )
        return vec / vec.sum()

    avged = []
    for layer in attn:
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        vec = torch.concat(
            [
                torch.tensor([0.0]),  # Zero out null attention on <bos>
                attns_per_head[-1][1:].cpu(),
                torch.tensor([0.0]),  # Zero for final generated token
            ]
        )
        avged.append(vec / vec.sum())
    return torch.stack(avged).mean(dim=0)

def heterogenous_stack(vecs: list[torch.Tensor]) -> torch.Tensor:
    """Pad vectors with zeros to same length then stack."""
    max_length = max(v.shape[0] for v in vecs)
    return torch.stack([torch.concat((v, torch.zeros(max_length - v.shape[0]))) for v in vecs])

def aggregate_prompt_attention(
    first_generated_token_attn: Tensor,
    layer_idx: int | None = None,
) -> Tensor:
    """
    Aggregate attention for the prompt/first generated token.

    Args:
        first_generated_token_attn: Attention tensor from first generation step
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
    """
    if layer_idx is not None and layer_idx != -100:
        layer_attn = first_generated_token_attn[layer_idx]
        layer_attns = layer_attn.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        # last row corresponds to the generated token, exclude it
        cur = attns_per_head[:-1].cpu().clone()

        # Zero out attention to the first <bos> token (null attention)
        # Skip first row since <bos> is the only token it can attend to
        cur[1:, 0] = 0.0
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)

        return cur

    aggregated = []
    for layer_attn in first_generated_token_attn:
        layer_attns = layer_attn.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        # last row corresponds to the generated token, exclude it
        cur = attns_per_head[:-1].cpu().clone()

        # Zero out attention to the first <bos> token (null attention)
        # Skip first row since <bos> is the only token it can attend to
        cur[1:, 0] = 0.0
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)

        aggregated.append(cur)

    # Average across all layers
    return torch.stack(aggregated).mean(dim=0)

def build_attention_matrix(outputs, layer_idx: int | None = None, valid_token_indices=None) -> torch.Tensor:
    """
    Build a complete attention matrix from generation outputs.

    Args:
        outputs: Model generation outputs with attentions
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
        valid_token_indices: Optional set of valid token indices to include in the attention matrix.

    Returns:
        Tensor of shape (total_tokens, max_seq_len) representing attention
        across all tokens (prompt + generated)
        entry (i, j): how much token i attends to token j
    """
    attention_vectors = []

    # Add <bos> token (attends only to itself)
    attention_vectors.append(torch.tensor([1.0]))

    # Add prompt token attentions
    prompt_attention = aggregate_prompt_attention(outputs.attentions[0], layer_idx=layer_idx)
    attention_vectors.extend(list(prompt_attention))

    # Add generated token attentions
    for step_idx, step_attention in enumerate(outputs.attentions):
        if valid_token_indices is not None and step_idx not in valid_token_indices:
            continue
        gen_attention = aggregate_llm_attention(step_attention, layer_idx=layer_idx)
        attention_vectors.append(gen_attention)

    # Pad all vectors to same length and stack
    return heterogenous_stack(attention_vectors)

def get_token_img_shape(inputs, processor, vlm_type="qwen3_vl"):
    if vlm_type == "qwen3_vl":
        """
        Qwen3-VL: Calculate the grid dimensions (height, width) of image tokens after spatial merging.
        NOTE: Processor outputs containing 'image_grid_thw'.
        """
        # Get the raw grid from the processor inputs
        grid_h = inputs["image_grid_thw"][0][1].item()
        grid_w = inputs["image_grid_thw"][0][2].item()

        # 2x2 spatial merging/pooling
        merge_factor = processor.image_processor.merge_size
        grid_h = grid_h // merge_factor
        grid_w = grid_w // merge_factor
        return grid_h, grid_w

STOP_WORDS = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
}

def clean_token(token: str) -> str:
    """Lowercase, strip, and drop punctuation to normalize tokens."""
    return token.strip().lower().translate(str.maketrans("", "", string.punctuation))

def is_content_word(token: str) -> bool:
    """Return True if token is non-empty after cleaning and not a stop word."""
    cleaned = clean_token(token)
    return bool(cleaned) and cleaned not in STOP_WORDS

def filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:
    """
    Return indices of tokens to use. If use_content_words is True, keep only content
    words; otherwise return all indices. Falls back to all tokens if filtering empties.
    """
    if not use_content_words:
        return list(range(len(token_ids)))

    logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")
    valid_indices = []
    for idx, token_id in enumerate(token_ids):
        token_str = tokenizer.decode([token_id], skip_special_tokens=False)
        if is_content_word(token_str):
            valid_indices.append(idx)
        elif DEBUG:
            logger.debug(f"Filtering out token '{token_str}' at index {idx}")

    if not valid_indices:
        return list(range(len(token_ids)))

    return valid_indices


### Qwen attention — CARA concept-token wrapper

The SAGA functions above retain causal attention-row alignment. This wrapper groups rows by FINER concepts rather than averaging all content words.

In [ ]:
class QwenAttentionXAI:
    def __init__(self, captioner, layer=29):
        self.captioner = captioner
        self.layer = layer

    @staticmethod
    def concept_offsets(tokenizer, generated_ids, concepts):
        ids = [int(x) for x in generated_ids]
        prefixes = [tokenizer.decode(ids[:i], skip_special_tokens=False) for i in range(len(ids) + 1)]
        spans = [(len(prefixes[i]), len(prefixes[i + 1])) for i in range(len(ids))]
        generated = prefixes[-1]
        result = {}
        for concept in concepts:
            words = [word for word in re.findall(r"\w+", concept.lower()) if is_content_word(word)]
            matches = [m.span() for m in re.finditer(r"\w+", generated.lower())
                       if m.group() in words]
            offsets = [i for i, (start, end) in enumerate(spans)
                       if any(start < b and end > a for a, b in matches)]
            if offsets:
                result[concept] = offsets
        return result

    @torch.no_grad()
    def maps(self, image, concepts):
        captioner = self.captioner
        inputs = captioner.attention_inputs(image)
        outputs = captioner.model.generate(
            **inputs,
            max_new_tokens=50,
            return_dict_in_generate=True,
            output_attentions=True,
            do_sample=False,
        )
        decoded = captioner.processor.batch_decode(outputs.sequences,
                      skip_special_tokens=False, clean_up_tokenization_spaces=False)[0]
        indices = get_token_indices(decoded, captioner.processor.tokenizer, outputs, inputs, "qwen3_vl")
        generated = outputs.sequences[0][indices["output_start"]:indices["output_end"]]
        offsets = self.concept_offsets(captioner.processor.tokenizer, generated, concepts)
        content = set(filter_generated_tokens(captioner.processor.tokenizer, generated, True))
        attention = build_attention_matrix(outputs, layer_idx=self.layer)
        height, width = get_token_img_shape(inputs, captioner.processor, "qwen3_vl")
        start, end = indices["vision_start"], indices["vision_end"]
        if end - start != height * width:
            raise ValueError("Vision-token count does not match Qwen image grid")

        concept_maps = {}
        for concept, positions in offsets.items():
            rows = []
            for position in positions:
                row_index = indices["output_start"] + position
                if position not in content or row_index >= attention.shape[0]:
                    continue
                weights = attention[row_index, start:end].float()
                if torch.isfinite(weights).all() and weights.sum() > 0:
                    rows.append((weights / weights.sum()).reshape(height, width))
            if rows:
                concept_maps[concept] = torch.stack(rows).mean(0)

        caption = captioner.processor.tokenizer.decode(generated, skip_special_tokens=True)
        if not concept_maps:
            raise ValueError("No generated Qwen tokens matched source concepts")
        return concept_maps, {"generated_caption": caption, "layer": self.layer}


## Feature Extractors

### CLIP ViT-g/14

In [ ]:
class CLIPG14:
    def __init__(self, device):
        from transformers import CLIPModel, AutoTokenizer
        self.device = torch.device(device)
        self.model = CLIPModel.from_pretrained(CLIP_G14_ID).eval().to(self.device)
        self.model.requires_grad_(False)
        self.tokenizer = AutoTokenizer.from_pretrained(CLIP_G14_ID)

    def encode_image(self, image):
        pixels = F.interpolate(image.to(self.device).float(), (224, 224), mode="bicubic", align_corners=False)
        pixels = (pixels / 255).clamp(0, 1)
        mean = pixels.new_tensor([0.48145466, 0.4578275, 0.40821073])[None, :, None, None]
        std = pixels.new_tensor([0.26862954, 0.26130258, 0.27577711])[None, :, None, None]
        return l2_normalize(self.model.get_image_features(pixel_values=(pixels - mean) / std))

    @torch.no_grad()
    def encode_text(self, captions):
        tokens = self.tokenizer(captions, padding=True, return_tensors="pt").to(self.device)
        if tokens.input_ids.shape[1] > self.model.config.text_config.max_position_embeddings:
            raise ValueError("Caption exceeds CLIP token length")
        return l2_normalize(self.model.get_text_features(**tokens))


### CLIP ViT-B/16

In [ ]:
class CLIPB16:
    def __init__(self, device):
        from transformers import CLIPModel, AutoTokenizer
        self.device = torch.device(device)
        self.model = CLIPModel.from_pretrained(CLIP_B16_ID).eval().to(self.device)
        self.model.requires_grad_(False)
        self.tokenizer = AutoTokenizer.from_pretrained(CLIP_B16_ID)

    def encode_image(self, image):
        pixels = F.interpolate(image.to(self.device).float(), (224, 224), mode="bicubic", align_corners=False)
        pixels = (pixels / 255).clamp(0, 1)
        mean = pixels.new_tensor([0.48145466, 0.4578275, 0.40821073])[None, :, None, None]
        std = pixels.new_tensor([0.26862954, 0.26130258, 0.27577711])[None, :, None, None]
        return l2_normalize(self.model.get_image_features(pixel_values=(pixels - mean) / std))

    @torch.no_grad()
    def encode_text(self, captions):
        tokens = self.tokenizer(captions, padding=True, return_tensors="pt").to(self.device)
        if tokens.input_ids.shape[1] > self.model.config.text_config.max_position_embeddings:
            raise ValueError("Caption exceeds CLIP token length")
        return l2_normalize(self.model.get_text_features(**tokens))


### CLIP ViT-B/32

In [ ]:
class CLIPB32:
    def __init__(self, device):
        from transformers import CLIPModel, AutoTokenizer
        self.device = torch.device(device)
        self.model = CLIPModel.from_pretrained(CLIP_B32_ID).eval().to(self.device)
        self.model.requires_grad_(False)
        self.tokenizer = AutoTokenizer.from_pretrained(CLIP_B32_ID)

    def encode_image(self, image):
        pixels = F.interpolate(image.to(self.device).float(), (224, 224), mode="bicubic", align_corners=False)
        pixels = (pixels / 255).clamp(0, 1)
        mean = pixels.new_tensor([0.48145466, 0.4578275, 0.40821073])[None, :, None, None]
        std = pixels.new_tensor([0.26862954, 0.26130258, 0.27577711])[None, :, None, None]
        return l2_normalize(self.model.get_image_features(pixel_values=(pixels - mean) / std))

    @torch.no_grad()
    def encode_text(self, captions):
        tokens = self.tokenizer(captions, padding=True, return_tensors="pt").to(self.device)
        if tokens.input_ids.shape[1] > self.model.config.text_config.max_position_embeddings:
            raise ValueError("Caption exceeds CLIP token length")
        return l2_normalize(self.model.get_text_features(**tokens))


### Qwen3-VL — Caption generation, FINER prompts and attention inputs

In [ ]:
class QwenVL:
    def __init__(self, model_id=QWEN_ID, attention=False):
        from transformers import AutoModelForImageTextToText, AutoProcessor

        self.processor = AutoProcessor.from_pretrained(model_id)
        device_count = torch.cuda.device_count()
        if device_count == 0:
            raise RuntimeError("Qwen requires CUDA GPUs")
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        memory = {}
        for gpu in range(device_count):
            free_bytes, _ = torch.cuda.mem_get_info(gpu)
            free_gib = free_bytes / 2 ** 30
            limit = QWEN_MAX_MEMORY_GIB.get(gpu, int(free_gib))
            available = min(float(limit), free_gib - 1.0)
            if available > 1:
                memory[gpu] = f"{available:.1f}GiB"
        if not memory:
            raise RuntimeError("Insufficient CUDA memory to load Qwen")

        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            torch_dtype=dtype,
            attn_implementation="eager" if attention else "sdpa",
            device_map="auto",
            max_memory=memory,
        ).eval()
        self.model.requires_grad_(False)

    def prepare(self, messages, images=None):
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch = self.processor(text=[text], images=images, return_tensors="pt")
        return batch.to(self.model.device)

    @torch.no_grad()
    def generate(self, text_or_messages, image=None, sample=False, max_new_tokens=80):
        if isinstance(text_or_messages, str):
            batch = self.processor(text=[text_or_messages], return_tensors="pt").to(self.model.device)
        else:
            batch = self.prepare(text_or_messages, images=[image] if image is not None else None)
        kwargs = {"do_sample": sample, "max_new_tokens": max_new_tokens}
        if sample:
            kwargs.update(temperature=CAPTION_TEMPERATURE, top_p=CAPTION_TOP_P)
        output = self.model.generate(**batch, **kwargs)
        generated = output[:, batch["input_ids"].shape[1]:]
        return self.processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

    def caption(self, image, sample=True):
        messages = [{
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": EVALUATION_PROMPT}],
        }]
        return self.generate(messages, image=image, sample=sample, max_new_tokens=50)

    def attention_inputs(self, image):
        # Same Qwen vision-token preprocessing concept as SAGA, using a PIL image directly.
        from qwen_vl_utils import process_vision_info
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Describe this image, no longer than 25 words."},
            ],
        }]
        images, videos, _ = process_vision_info(
            messages,
            image_patch_size=16,
            return_video_kwargs=True,
            return_video_metadata=True,
        )
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch = self.processor(text=text, images=images, videos=videos,
                               do_resize=False, return_tensors="pt")
        return batch.to(self.model.device)


### CLIP ensemble — Target selection and fusion

In [ ]:
class CLIPEnsemble:
    def __init__(self):
        self.device = torch.device(ATTACK_DEVICE)
        self.models = [
            CLIPG14(CLIP_DEVICES["G14"]),
            CLIPB16(CLIP_DEVICES["B16"]),
            CLIPB32(CLIP_DEVICES["B32"]),
        ]

    def encode_image(self, image):
        features = [model.encode_image(image).to(self.device) for model in self.models]
        return torch.cat(features, dim=-1)

    @torch.no_grad()
    def encode_text(self, captions):
        return torch.cat([model.encode_text(captions).to(self.device)
                          for model in self.models], dim=-1)

    @torch.no_grad()
    def anchors(self, image, captions):
        features = []
        for model in self.models:
            image_feature = model.encode_image(image).to(self.device)
            text_features = model.encode_text(captions).to(self.device)
            fused = IMAGE_WEIGHT * image_feature + TEXT_WEIGHT * text_features
            features.append(fused)
        return torch.cat(features, dim=-1)

    @torch.no_grad()
    def build_targets(self, source, target, bank):
        target_bank = self.anchors(target, bank["target_captions"])
        negative_bank = self.anchors(source, bank["negative_captions"])
        distances = (target_bank - target_bank.mean(dim=0, keepdim=True)).square().sum(dim=-1)
        selected_index = int(distances.argmin())
        positive = target_bank[selected_index:selected_index + 1]
        if negative_bank.shape[0] != K:
            raise ValueError("The loss requires K negative anchors")
        return positive.detach(), negative_bank.detach(), selected_index


## Loss Function

### MPCAttack — Original InfoNCE implementation

Kept verbatim as a direct function, for source comparison. The next class extends this from one source vector to K negatives.

In [ ]:
def info_nce_loss(feat_adv, feat_tgt, feat_src, temperature=0.2, mode='img', omega=2.0):
    """
    InfoNCE loss for pulling adv->tgt close, pushing adv->src away.

    Args:
        feat_adv: Tensor [B, D] adversarial features
        feat_tgt: Tensor [B, D] target features (positive)
        feat_src: Tensor [B, D] source features (negative)
        temperature: float, softmax temperature

    Returns:
        loss: scalar tensor
    """
    # L2 normalize
    feat_adv = F.normalize(feat_adv, dim=-1)
    feat_tgt = F.normalize(feat_tgt, dim=-1)
    feat_src = F.normalize(feat_src, dim=-1)
    # Cosine similarities
    if mode == 'text':
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1).mean(dim=1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1).mean(dim=1) / temperature  # [B]
    else:
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1) / temperature  # [B]
    # Stack positive and negative: shape [B, 2]
    logits = torch.stack([sim_pos, sim_neg], dim=1)
    labels = torch.zeros(feat_adv.size(0), dtype=torch.long, device=feat_adv.device)  # positives index=0
    # Cross-entropy loss
    loss = F.cross_entropy(logits, labels)
    return - loss


### CARA — K-negative contrastive loss

In [ ]:
class ContrastiveLoss:
    def __init__(self, temperature=0.2, omega=2.0):
        self.temperature = temperature
        self.omega = omega

    def __call__(self, adversarial, positive, negatives):
        query = l2_normalize(adversarial)
        positive = l2_normalize(positive)
        negatives = l2_normalize(negatives)
        if query.ndim != 2 or positive.shape != query.shape:
            raise ValueError("Expected query and positive of shape [B, D]")
        if negatives.ndim != 2 or negatives.shape[1] != query.shape[1]:
            raise ValueError("Expected K negative anchors of shape [K, D]")

        positive_sim = (query * positive).sum(dim=-1, keepdim=True)
        negative_sim = query @ negatives.T
        logits = torch.cat([self.omega * positive_sim, negative_sim], dim=-1)
        labels = torch.zeros(query.shape[0], dtype=torch.long, device=query.device)
        cross_entropy = F.cross_entropy(logits / self.temperature, labels)
        return -cross_entropy, positive_sim.mean(), negative_sim.max()


## Attack

### CARAAttack — Paper-style attack loop

Like MPCAttack: initialize perturbation → create anchors → crop → compute surrogate features → InfoNCE → `torch.autograd.grad` → momentum sign ascent → projection. Region masking and dynamic XAI are CARA-specific.

In [ ]:
class CARAAttack:
    def __init__(self, encoder, explainer, output_dir):
        self.encoder = encoder
        self.explainer = explainer
        self.loss_function = ContrastiveLoss(TEMPERATURE, OMEGA)
        self.selector = RegionSelector(IMAGE_SIZE, IMAGE_SIZE)
        self.output_dir = Path(output_dir)

    def save_xai(self, image, concepts, sample_dir, role, step, previous=None):
        xai_dir = sample_dir / "xai"
        xai_dir.mkdir(parents=True, exist_ok=True)
        info = {"step": step, "role": role, "method": XAI_METHOD, "valid": False}
        try:
            maps, metadata = self.explainer.maps(tensor_to_image(image), concepts)
            heatmap, per_concept = XAIMap.normalize(maps)
            info.update(valid=True, dispersion=XAIMap.dispersion(heatmap), metadata=metadata)
            data = {"aggregate": heatmap.numpy()}
            for index, (concept, value) in enumerate(per_concept.items()):
                data[f"concept_{index}"] = value.numpy()
            np.savez_compressed(xai_dir / f"{role}_{step:03d}.npz", **data)
            info["concept_labels"] = list(per_concept)
        except Exception as error:
            info["error"] = f"{type(error).__name__}: {error}"
            heatmap = previous
            info["fallback"] = heatmap is not None
            if heatmap is not None:
                np.savez_compressed(xai_dir / f"{role}_{step:03d}.npz", aggregate=heatmap.numpy())
        write_json(xai_dir / f"{role}_{step:03d}.json", info)
        return heatmap, info

    def attack(self, source, target, caption_bank, sample_id):
        sample_dir = self.output_dir / f"{sample_id:05d}"
        sample_dir.mkdir(parents=True, exist_ok=True)
        source = source.to(ATTACK_DEVICE)
        target = target.to(ATTACK_DEVICE)
        positive, negatives, selected = self.encoder.build_targets(source, target, caption_bank)
        delta = torch.zeros_like(source, requires_grad=True)
        momentum = torch.zeros_like(source)
        history = []
        xai_history = []

        write_json(sample_dir / "bank.json", caption_bank)
        write_json(sample_dir / "anchors.json", {
            "selected_target": selected,
            "target_caption": caption_bank["target_captions"][selected],
        })
        with torch.no_grad():
            caption_groups = ["source_captions", "target_captions", "negative_captions"]
            caption_vectors = {
                name: self.encoder.encode_text(caption_bank[name]).detach().cpu().numpy()
                for name in caption_groups
            }
            np.savez_compressed(sample_dir / "caption_embeddings.npz", **caption_vectors)
        step_dir = sample_dir / "steps"
        step_dir.mkdir(exist_ok=True)
        tensor_to_image(source).save(sample_dir / "source.png")
        tensor_to_image(target).save(sample_dir / "target.png")
        tensor_to_image(source).save(step_dir / "step_000.png")

        source_concepts = caption_bank["source_concepts"]
        target_concepts = caption_bank["target_concepts"]
        _, _ = self.save_xai(target, target_concepts, sample_dir, "target", 0)
        heat, initial_xai = self.save_xai(source, source_concepts, sample_dir, "source", 0)
        xai_history.append(initial_xai)
        if heat is None:
            raise RuntimeError("No valid source XAI map; cannot start guided cropping")

        # At step 0 the adversarial image equals the source image.
        np.savez_compressed(sample_dir / "xai" / "adversarial_000.npz", aggregate=heat.numpy())
        write_json(sample_dir / "xai" / "adversarial_000.json", {
            "step": 0,
            "role": "adversarial",
            "valid": True,
            "initially_equals_source": True,
            "dispersion": XAIMap.dispersion(heat),
        })

        crop_box = None
        for step in tqdm(range(1, STEPS + 1), desc=f"CARA sample {sample_id}"):
            if crop_box is None or step >= RANDOM_START:
                top, left, height, width, mode = self.selector.select(heat, step)
                crop_box = (top, left, height, width, mode)
            top, left, height, width, mode = crop_box

            adversarial = (source + delta).clamp(0, 255)
            cropped = adversarial[:, :, top:top + height, left:left + width]
            features = self.encoder.encode_image(cropped)
            score, positive_sim, negative_sim = self.loss_function(features, positive, negatives)
            gradient = torch.autograd.grad(score, delta)[0]
            if not torch.isfinite(gradient).all() or gradient.abs().sum() == 0:
                raise RuntimeError(f"Invalid gradient at step {step}")

            momentum = MOMENTUM * momentum + gradient
            update = ALPHA * momentum.sign()
            if STRICT_CROP_MASK:
                mask = torch.zeros_like(update)
                mask[:, :, top:top + height, left:left + width] = 1
                update = update * mask

            with torch.no_grad():
                delta = (delta + update).clamp(-EPSILON, EPSILON)
                delta = torch.max(torch.min(delta, 255 - source), -source)
            delta.requires_grad_(True)

            history.append({
                "step": step,
                "loss": float(-score.detach()),
                "target_similarity": float(positive_sim.detach()),
                "max_negative_similarity": float(negative_sim.detach()),
                "mode": mode,
                "crop": [top, left, height, width],
            })

            if step % SAVE_EVERY == 0:
                current = (source + delta).clamp(0, 255)
                tensor_to_image(current).save(step_dir / f"step_{step:03d}.png")
                heat_new, info = self.save_xai(current, source_concepts,
                                               sample_dir, "adversarial", step, previous=heat)
                xai_history.append(info)
                if heat_new is not None:
                    heat = heat_new
                crop_box = None
                write_json(sample_dir / "progress.json", {"history": history, "xai": xai_history})
                print(f"Step {step:03d} | CE: {history[-1]['loss']:.4f} | mode: {mode}")

        result = (source + delta).clamp(0, 255)
        tensor_to_image(result).save(sample_dir / "adversarial.png")
        difference = (result.detach() - source).abs().max().item()
        if difference > EPSILON + 1e-5:
            raise AssertionError(f"L-infinity bound violated: {difference}")
        write_json(sample_dir / "summary.json", {
            "sample_id": sample_id,
            "updates": STEPS,
            "linf_pixels": difference,
            "target_caption": caption_bank["target_captions"][selected],
        })
        return result.detach().cpu()


### Experiment Runner — 5-sample pilot and 200-sample expansion

In [ ]:
class Experiment:
    def __init__(self, dataset, output_dir):
        self.dataset = dataset
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def generate_banks(self, pairs):
        captioner = QwenVL(QWEN_ID, attention=False)
        builder = CaptionBank(captioner, K)
        results = {}
        for index, (source_path, target_path) in enumerate(pairs, start=START_INDEX):
            bank_file = self.output_dir / f"{index:05d}" / "bank.json"
            if bank_file.exists():
                results[index] = read_json(bank_file)
                continue
            source = self.dataset.load_image(source_path)
            target = self.dataset.load_image(target_path)
            bank = builder.create(source, target)
            results[index] = bank
            write_json(bank_file, bank)
            print(f"Caption bank {index}: {K} target / {K} source / {K} negatives")
        del captioner, builder
        release_memory()
        return results

    def run(self, pairs):
        banks = self.generate_banks(pairs)
        encoder = CLIPEnsemble()
        if XAI_METHOD == "grad_eclip":
            explainer = GradEclipXAI(ATTACK_DEVICE)
        elif XAI_METHOD == "qwen_attention":
            captioner = QwenVL(QWEN_ID, attention=True)
            explainer = QwenAttentionXAI(captioner, ATTENTION_LAYER)
        else:
            raise ValueError(f"Unknown XAI method: {XAI_METHOD}")

        attack = CARAAttack(encoder, explainer, self.output_dir)
        records = []
        for index, (source_path, target_path) in enumerate(pairs, start=START_INDEX):
            set_seed(SEED + index)
            try:
                source = image_to_tensor(self.dataset.load_image(source_path))
                target = image_to_tensor(self.dataset.load_image(target_path))
                attack.attack(source, target, banks[index], index)
                records.append({"sample": index, "status": "completed"})
            except Exception as error:
                records.append({"sample": index, "status": "failed", "error": str(error)})
                print(f"Sample {index} failed: {error}")
            write_json(self.output_dir / "status.json", records)
        del encoder, explainer, attack
        release_memory()
        return pd.DataFrame(records)


## Evaluation

In [ ]:
class CaptionEvaluator:
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)

    def evaluate(self, sample_ids):
        captioner = QwenVL(EVALUATION_ID, attention=False)
        rows = []
        for index in sample_ids:
            folder = self.output_dir / f"{index:05d}"
            image_path = folder / "adversarial.png"
            if not image_path.exists():
                continue
            with Image.open(folder / "source.png") as handle:
                source = handle.convert("RGB")
            with Image.open(image_path) as handle:
                adversarial = handle.convert("RGB")
            bank = read_json(folder / "bank.json")
            rows.append({
                "sample": index,
                "source_caption": captioner.caption(source, sample=False),
                "adversarial_caption": captioner.caption(adversarial, sample=False),
                "target_caption": read_json(folder / "anchors.json")["target_caption"],
                "target_success": "",   # Fill manually after checking images and captions.
            })
        frame = pd.DataFrame(rows)
        frame.to_csv(self.output_dir / "human_review.csv", index=False)
        del captioner
        release_memory()
        return frame


## Visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


class CARAVisualization:
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)

    def folder(self, sample_id):
        return self.output_dir / f"{sample_id:05d}"

    def steps(self, sample_id):
        folder = self.folder(sample_id) / "steps"
        files = sorted(folder.glob("step_*.png"))
        fig, axes = plt.subplots(2, math.ceil(len(files) / 2), figsize=(18, 7))
        for axis, path in zip(axes.ravel(), files):
            axis.imshow(Image.open(path))
            axis.set_title(path.stem)
            axis.axis("off")
        for axis in axes.ravel()[len(files):]:
            axis.axis("off")
        plt.tight_layout()
        plt.show()

    def attention(self, sample_id, role="adversarial"):
        files = sorted((self.folder(sample_id) / "xai").glob(f"{role}_*.npz"))
        if not files:
            print(f"No {role} XAI maps saved")
            return
        fig, axes = plt.subplots(2, math.ceil(len(files) / 2), figsize=(18, 6))
        for axis, path in zip(axes.ravel(), files):
            array = np.load(path)["aggregate"]
            axis.imshow(array, cmap="magma")
            axis.set_title(path.stem)
            axis.axis("off")
        for axis in axes.ravel()[len(files):]:
            axis.axis("off")
        plt.tight_layout()
        plt.show()

    def overlay(self, sample_id, role="source", step=0):
        folder = self.folder(sample_id)
        image_file = folder / f"{role}.png"
        if role == "adversarial":
            image_file = folder / "steps" / f"step_{step:03d}.png"
        heat_file = folder / "xai" / f"{role}_{step:03d}.npz"
        image = Image.open(image_file).convert("RGB")
        heat = torch.from_numpy(np.load(heat_file)["aggregate"])
        heat = F.interpolate(heat[None, None], size=(image.height, image.width),
                             mode="bilinear", align_corners=False)[0, 0].numpy()
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(image)
        axes[1].imshow(heat, cmap="magma")
        axes[2].imshow(image)
        axes[2].imshow(heat, cmap="magma", alpha=0.50)
        for ax, title in zip(axes, ["Image", "XAI heatmap", "Overlay"]):
            ax.set_title(title)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    def source_target_attention(self, sample_id):
        self.overlay(sample_id, "source", 0)
        self.overlay(sample_id, "target", 0)

    def similarity(self, sample_id):
        progress = read_json(self.folder(sample_id) / "progress.json")
        frame = pd.DataFrame(progress["history"])
        ax = frame.plot(x="step", y=["target_similarity", "max_negative_similarity"], figsize=(10, 4))
        ax.set_ylabel("Cosine similarity")
        plt.show()
        ax = frame.plot(x="step", y="loss", figsize=(10, 4))
        ax.set_ylabel("InfoNCE cross-entropy")
        plt.show()

    def dispersion(self, sample_id):
        files = sorted((self.folder(sample_id) / "xai").glob("adversarial_*.json"))
        rows = []
        for path in files:
            record = read_json(path)
            if record.get("dispersion"):
                rows.append({"step": record["step"], **record["dispersion"]})
        if rows:
            pd.DataFrame(rows).plot(x="step", y=["entropy", "effective_support", "variance"],
                                    figsize=(10, 4), title="Attention dispersion")
            plt.show()

    def caption_space(self, sample_id):
        path = self.folder(sample_id) / "caption_embeddings.npz"
        bank = read_json(self.folder(sample_id) / "bank.json")
        groups = ["source_captions", "target_captions", "negative_captions"]
        data = np.load(path)
        vectors = np.concatenate([data[name] for name in groups], axis=0)
        points = PCA(n_components=2).fit_transform(vectors)
        for group_index, name in enumerate(groups):
            subset = points[group_index * K:(group_index + 1) * K]
            plt.scatter(subset[:, 0], subset[:, 1], label=name)
            for i, (x, y) in enumerate(subset):
                plt.annotate(str(i + 1), (x, y), fontsize=8)
        plt.legend()
        plt.title("CLIP caption embeddings — PCA")
        plt.show()


## Validation

In [ ]:
def validate_logic():
    assert RegionSelector.stage_area(1) == 0.5
    assert RegionSelector.stage_area(300) == 1.0
    assert RegionSelector.stage_area(150) == 13 / 18
    assert RegionSelector.stage_area(271) == 1.0
    assert XAI_METHOD in {"grad_eclip", "qwen_attention"}
    assert K == 5 and STEPS == 300 and SAVE_EVERY == 30
    assert _validate_single_replacement("a dog and a tree", "a cat and a tree")
    assert not _validate_single_replacement("a dog and a tree", "a cat and a bush")

    query = torch.randn(1, 16, requires_grad=True)
    positive = torch.randn(1, 16)
    negatives = torch.randn(K, 16)
    score, _, _ = ContrastiveLoss()(query, positive, negatives)
    gradient = torch.autograd.grad(score, query)[0]
    assert torch.isfinite(gradient).all()

    # K=1 must agree with the negative CE returned by original MPCAttack.
    one = negatives[:1]
    current_score, _, _ = ContrastiveLoss()(query, positive, one)
    expected = info_nce_loss(query, positive, one, temperature=TEMPERATURE, omega=OMEGA)
    torch.testing.assert_close(current_score, expected)

    print("Logic checks passed: FINER validation, 300-step schedule, K-negative InfoNCE, MPC K=1.")


validate_logic()


## Run — Kaggle pilot

In [ ]:
PAIRS = []
RESULTS = None

if RUN_ATTACK:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable the GPU accelerator in Kaggle settings")
    PAIRS = DATASET.get_pairs(START_INDEX, NUM_SAMPLES)
    print(pd.DataFrame(PAIRS, columns=["source", "target"]))
    runner = Experiment(DATASET, OUTPUT_DIR)
    RESULTS = runner.run(PAIRS)
    display(RESULTS)


In [ ]:
REVIEW = None
if RUN_ATTACK and RUN_EVALUATION and RESULTS is not None:
    completed = RESULTS.loc[RESULTS["status"] == "completed", "sample"].tolist()
    if completed:
        REVIEW = CaptionEvaluator(OUTPUT_DIR).evaluate(completed)
        display(REVIEW)


In [ ]:
if RUN_ATTACK and RUN_VISUALIZATION and RESULTS is not None:
    completed = RESULTS.loc[RESULTS["status"] == "completed", "sample"].tolist()
    if completed:
        visualizer = CARAVisualization(OUTPUT_DIR)
        visualizer.steps(completed[0])
        visualizer.source_target_attention(completed[0])
        visualizer.attention(completed[0], role="adversarial")
        visualizer.similarity(completed[0])
        visualizer.dispersion(completed[0])
        visualizer.caption_space(completed[0])
